# Which genes switch on when a mouse makes milk?

The **mammary gland** changes enormously during pregnancy and when the mother feeds her young (**lactation**). It is built from two kinds of cells:

* **luminal cells** line the inside of the milk ducts and, during lactation, **produce the milk**,
* **basal cells** sit around them and **squeeze** the milk out, like a muscle.

Researchers (Fu et al. 2015) took both cell types from mice in three states: **virgin** (never pregnant), **pregnant** and **lactating**, with two mice per combination. They sequenced the RNA of these **12 samples**.

### Our research question

> Which genes do the **luminal cells** switch on (or off) when they go from **pregnancy** to **making milk**? Can we explain what milk is made of from these genes?

### The plan

```mermaid
flowchart LR
    A[1 Look at<br>the data] --> B[2 Define<br>groups] --> C[3 Prepare:<br>filter, normalise] --> D[4 Design and<br>contrasts] --> E[5 Test<br>with limma] --> F[6 Interpret]
```

Run the cells one after the other, from the top, with **Shift + Enter**.

In [ ]:
setLanguage("en")   # language of the messages and plots below

## 1 Look at the data

`loadDataset` loads the data. It contains three tables: `samples` (what each sample is), `counts` (the reads) and `genes` (gene names). Let's look at the samples first.

In [ ]:
data <- loadDataset("milk")
data$samples

Each sample has a name (`MCL1-DG` …), a cell type (`celltype`) and a state (`state`). The names in the table are in English, so `lactating` means "säugend" and `pregnant` "trächtig".

Now the counts. `head()` shows only the first 6 rows; the full table is much longer.

In [ ]:
head(data$counts)

Each **row** is a gene (the numbers on the left are gene IDs from the NCBI database), each **column** a sample, and each number says how many reads of that gene were found in that sample. The `genes` table translates the IDs into gene names:

In [ ]:
head(data$genes)

**Exercise:** How many genes and how many samples are in the count table? `dim()` gives the size of a table: first the number of rows, then of columns.

In [ ]:
dim(___)

✏️ **Your answer:** *(double-click here to type)*



For the analysis, we put the three tables into one container, a `DGEList` from the package **edgeR**. It also adds up the reads of each sample: the **library size**.

In [ ]:
dge <- DGEList(counts = data$counts, samples = data$samples, genes = data$genes)
dge$samples

In [ ]:
plotLibrarySizes(dge, colour_by = "celltype")

**Question:** Do all samples have the same number of reads? Why would it be wrong to compare the raw counts of a gene between two samples?

✏️ **Your answer:** *(double-click here to type)*



**Exercise:** Many genes are not used at all in these cells. How many genes have **0 reads in all 12 samples**? `rowSums()` adds up each row, and `sum(... == x)` counts how often something equals `x`.

In [ ]:
sum(rowSums(dge$counts) == ___)

✏️ **Your answer:** *(double-click here to type)*



## 2 Define the groups

To compare, limma has to know which samples belong together. Each combination of cell type and state is one **group**, for example `luminal_lactating`. We glue the two columns together with `paste` and store the result in the `group` column.

In [ ]:
group <- paste(dge$samples$celltype, dge$samples$state, sep = "_")
dge$samples$group <- factor(group, levels = unique(group))
table(dge$samples$group)

**Question:** How many samples does each group have? Why do researchers use several mice per group (**replicates**) instead of one?

✏️ **Your answer:** *(double-click here to type)*



Let's look at a single gene. **Csn2** is the gene for **β-casein**, the main protein in milk. `plotGene` shows its expression in every sample as **log2 counts per million**: going up by 1 means twice as much. Where do you expect it to be high?

In [ ]:
plotGene(dge, "Csn2", group_by = "group")

**Exercise:** Use your biology to check that the samples are labelled correctly. **Krt5** (keratin 5) is a typical protein of basal cells, **Krt8** (keratin 8) of luminal cells. Plot one of them.

In [ ]:
plotGene(dge, "___", group_by = "group")

✏️ **Your answer:** *(double-click here to type)*



## 3 Prepare the data

### Filtering

Genes with only a handful of reads can't be tested reliably: 0 vs. 2 reads could easily be chance. `filterByExpr` keeps only genes with enough reads in at least as many samples as the smallest group has (here 2). `TRUE` means keep.

In [ ]:
keep <- filterByExpr(dge)
table(keep)

In [ ]:
dge <- dge[keep, , keep.lib.sizes = FALSE]
dim(dge)

### Normalisation

Now let's look at the distribution of **all** genes in each sample. Each box shows the middle half of the genes; the line in the box is the median. If the samples are comparable, the boxes should be at about the same height.

In [ ]:
plotDistributions(dge, colour_by = "group", title = "Before normalisation")

Look at the two lactating luminal samples (`MCL1-LE`, `MCL1-LF`): their boxes are **lower**. Are all their genes really less active? No! These cells spend a huge part of their RNA on a few milk genes. Calculate the share of reads for Csn2 alone (gene ID 12991) in sample `MCL1-LE`:

In [ ]:
data$counts["12991", "MCL1-LE"] / sum(data$counts[, "MCL1-LE"])   # share of all reads

About **10 %** of all reads come from this one gene. Then fewer reads are left for all the other genes, and they *look* lower. The **TMM normalisation** in `normLibSizes` corrects this: it calculates a correction factor for each sample (`norm.factors`) from the many genes that don't change.

In [ ]:
dge <- normLibSizes(dge)
dge$samples[, c("group", "lib.size", "norm.factors")]

In [ ]:
plotDistributions(dge, colour_by = "group", title = "After normalisation")

**Question:** How many genes are left after filtering? Which samples get the smallest correction factor, and what changed in the box plot?

✏️ **Your answer:** *(double-click here to type)*



### The sample map (MDS)

Before testing single genes, let's see how similar the samples are **overall**. An **MDS plot** puts each sample on a map: samples whose ~16,000 genes are expressed similarly lie close together. **Distance 1** (left–right) is the direction in which the samples differ most; the percentage says how much of all differences it shows.

In [ ]:
plotSamples(dge, colour_by = "celltype", label_by = "state")

**Exercise:** Now colour the samples by state and label them by cell type.

In [ ]:
plotSamples(dge, colour_by = "___", label_by = "___")

**Question:** What separates the samples along distance 1, and what along distance 2? Do the two replicates of each group lie close together? Why is that good news?

✏️ **Your answer:** *(double-click here to type)*



## 4 Design and contrasts

### The design matrix

The **design matrix** tells limma which group each sample belongs to: one column per group, with a 1 if the sample belongs to it.

In [ ]:
design <- model.matrix(~0 + group, data = dge$samples)
colnames(design) <- levels(dge$samples$group)
design

### Contrasts: which groups do we compare?

A **contrast** is the comparison you are interested in, written as a subtraction: `A - B` asks "how is A different from B?". A **positive** log fold change then means **higher in A**.

Our question compares **lactating** with **pregnant** luminal cells. Why not simply all lactating with all pregnant samples? Because basal cells don't make milk, and mixing them in would blur the result: the MDS plot showed that the cell type is the biggest difference of all.

**Exercise:** Fill in the two contrasts:
1. `lactating_vs_pregnant`: luminal cells, lactating compared with pregnant,
2. `luminal_vs_basal`: in virgin mice, luminal compared with basal cells.

Use the group names exactly as in the design matrix.

In [ ]:
contrasts <- makeContrasts(
    lactating_vs_pregnant = ___ - ___,
    luminal_vs_basal      = ___ - ___,
    levels = design)
contrasts

**Question:** In `lactating_vs_pregnant`, what does a log fold change of **+3** mean for a gene? And **−1**? Which contrast would you write to find the genes that change in **basal** cells during pregnancy?

✏️ **Your answer:** *(double-click here to type)*



## 5 Test every gene with limma

Three steps:

1. **`voom`** turns the counts into log2 counts per million and works out how reliable each value is: genes with few reads are noisier, so they get less weight. The plot shows this: the fewer reads (left), the larger the variation (up).
2. **`lmFit`** calculates the average of every gene in every group, and **`contrasts.fit`** the differences we asked for in the contrasts.
3. **`eBayes`** tests whether each difference is larger than the natural variation between the replicates. With only 2 replicates that variation is hard to estimate for a single gene, so limma "borrows" information from all the other genes. That's what makes limma work with small experiments.

In [ ]:
v <- voom(dge, design, plot = TRUE)

In [ ]:
fit <- lmFit(v, design)
fit <- contrasts.fit(fit, contrasts)
fit <- eBayes(fit)

### How many genes changed?

For each gene, limma calculates a **p-value**: how likely is a difference this large if the gene didn't really change? Small = convincing.

But careful: we test about 16,000 genes. With the usual limit p < 0.05, about 5 % of the genes that **don't** change would still look significant by chance: that's 800 false hits! That's why limma also gives an **adjusted p-value** (`adj.P.Val`). If we take all genes with adjusted p < 0.05, only about 5 % of **this list** are false hits.

`decideTests` counts the genes with adjusted p < 0.05: up, down or not significant (`NotSig`).

In [ ]:
summary(decideTests(fit))

**Question:** How many genes go up and down from pregnancy to lactation? Which of the two contrasts has more changed genes, and does that fit the MDS plot?

✏️ **Your answer:** *(double-click here to type)*



## 6 Interpret the results

### The volcano plot

In a **volcano plot** every gene is a dot. Left–right is the log fold change (left: lower when lactating, right: higher), and up is how certain the result is. The most interesting genes are at the top left and top right: a big change **and** convincing. Coloured dots are significant and change at least 2-fold (dashed lines).

In [ ]:
plotVolcano(fit, "lactating_vs_pregnant")

### The top genes

`showTop` lists the most convincing genes with their name and description. `change` translates the log fold change back into "times higher/lower".

In [ ]:
showTop(fit, "lactating_vs_pregnant", n = 15, direction = "up")

**Exercise:** Choose a gene from the list that interests you and look at it in all groups.

Find out what it does: search for its name on [GeneCards](https://www.genecards.org) (for human genes; the mouse gene usually does the same), [NCBI Gene](https://www.ncbi.nlm.nih.gov/gene) or [UniProt](https://www.uniprot.org), or think about the description. *Tip: milk contains protein (caseins), fat, milk sugar (lactose), calcium, zinc and phosphate.*

In [ ]:
plotGene(dge, "___", group_by = "group")

**Question:** Can you connect the top genes to what milk is made of? Find at least three examples.

✏️ **Your answer:** *(double-click here to type)*



Where are the famous milk genes like **Wap** (whey acidic protein) and **Lalba** (α-lactalbumin, needed to make lactose)? Look at one of them:

In [ ]:
plotGene(dge, "Wap", group_by = "group")

**Question:** Wap is significantly higher in lactating cells, but it is not at the top of the list. Why not?

✏️ **Your answer:** *(double-click here to type)*



Now the genes that go **down**:

In [ ]:
showTop(fit, "lactating_vs_pregnant", n = 10, direction = "down")

### Your conclusion

**Exercise:** Answer the research question in two or three sentences, as you would in a lab report: what did you compare, what did you find, and what does it mean?

✏️ **Your answer:** *(double-click here to type)*



### Bonus: luminal vs basal cells

Look at the second contrast. Which genes make a basal cell a basal cell?

In [ ]:
showTop(fit, "___", n = 15)

✏️ **Your answer:** *(double-click here to type)*



**Well done!** 🎉 You've done a complete differential expression analysis, the same way researchers do it. Now it's your turn: choose one of the notebooks `02_diy_…`, `03_diy_…` or `04_diy_…` and answer a new research question on your own.

## Sources

**Data**
* **Milk (tutorial):** Fu NY, Rios AC, Pal B, et al. (2015). EGF-mediated induction of Mcl-1 at the switch to lactation is essential for alveolar cell survival. *Nature Cell Biology* 17:365–375. [doi:10.1038/ncb3117](https://doi.org/10.1038/ncb3117). Counts from GEO [GSE60450](https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE60450).
* **Dexamethasone (airway):** Himes BE, Jiang X, Wagner P, et al. (2014). RNA-Seq transcriptome profiling identifies CRISPLD2 as a glucocorticoid responsive gene that modulates cytokine function in airway smooth muscle cells. *PLoS ONE* 9(6):e99625. [doi:10.1371/journal.pone.0099625](https://doi.org/10.1371/journal.pone.0099625). Counts from the Bioconductor package [airway](https://bioconductor.org/packages/airway) (Love M).
* **Pasilla:** Brooks AN, Yang L, Duff MO, et al. (2011). Conservation of an RNA regulatory map between *Drosophila* and mammals. *Genome Research* 21:193–202. [doi:10.1101/gr.108662.110](https://doi.org/10.1101/gr.108662.110). Counts from the Bioconductor package [pasilla](https://bioconductor.org/packages/pasilla).
* **Breast cancer:** The Cancer Genome Atlas Network (2012). Comprehensive molecular portraits of human breast tumours. *Nature* 490:61–70. [doi:10.1038/nature11412](https://doi.org/10.1038/nature11412). Counts (STAR – Counts) of 10 TCGA-BRCA patients from the [NCI Genomic Data Commons](https://portal.gdc.cancer.gov).
* **Gene names:** Bioconductor annotation packages org.Mm.eg.db, org.Hs.eg.db, org.Dm.eg.db (Carlson M).

**Software and method**
* **limma:** Ritchie ME, Phipson B, Wu D, et al. (2015). limma powers differential expression analyses for RNA-sequencing and microarray studies. *Nucleic Acids Research* 43(7):e47. [doi:10.1093/nar/gkv007](https://doi.org/10.1093/nar/gkv007)
* **voom:** Law CW, Chen Y, Shi W, Smyth GK (2014). voom: precision weights unlock linear model analysis tools for RNA-seq read counts. *Genome Biology* 15:R29. [doi:10.1186/gb-2014-15-2-r29](https://doi.org/10.1186/gb-2014-15-2-r29)
* **edgeR:** Chen Y, Chen L, Lun ATL, Baldoni PL, Smyth GK (2025). edgeR v4: powerful differential analysis of sequencing data with expanded functionality and improved support for small counts and larger datasets. *Nucleic Acids Research* 53(2):gkaf018. [doi:10.1093/nar/gkaf018](https://doi.org/10.1093/nar/gkaf018)
* **Workflow this practical follows:** Law CW, Alhamdoosh M, Su S, et al. (2016). RNA-seq analysis is easy as 1-2-3 with limma, Glimma and edgeR. *F1000Research* 5:1408. [doi:10.12688/f1000research.9005.3](https://doi.org/10.12688/f1000research.9005.3)